# Seçmen Şapkası: Basit Bir Sınıflandırma Modeli

[![Colab'da Aç](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haticeozbolat01/pop-kultur-kodluyorum/blob/main/06-secmen-sapkasi.ipynb)

Bu not defteri, haticeozbolat.com'daki **Harry Potter'ın Seçmen Şapkası: Basit Bir Sınıflandırma Modeli** yazısının kodlarını içeriyor. Yazı: [Yazıyı oku]([YAZI LİNKİ])

**Veri hakkında:** Buradaki öğrencilerin hepsi hayali. Veriyi kod kendisi üretiyor; kitaplardan ya da filmlerden hiçbir karakter, metin veya görsel kullanılmıyor.

Hücreleri sırayla **Shift + Enter** ile çalıştır.

## 1. Kural tabanlı şapka

In [ ]:
OZELLIKLER = ["cesaret", "zeka", "sadakat", "hirs"]
BINALAR = {"Gryffindor": "cesaret", "Ravenclaw": "zeka",
           "Hufflepuff": "sadakat", "Slytherin": "hirs"}

def kural_sapka(ogrenci):
    """En yüksek puanlı özelliğe bakıp binayı seçer."""
    en_guclu = max(OZELLIKLER, key=lambda o: ogrenci[o])
    for bina, ozellik in BINALAR.items():
        if ozellik == en_guclu:
            return bina

print(kural_sapka({"cesaret": 9, "zeka": 6, "sadakat": 7, "hirs": 4}))
print(kural_sapka({"cesaret": 5.2, "zeka": 5.4, "sadakat": 5.1, "hirs": 5.3}))

## 2. Hayali öğrenci verisi üretmek

Her binadan 60 öğrenci. Gryffindor, Ravenclaw ve Slytherin öğrencilerinde binanın özelliği belirgin şekilde yüksek. Hufflepuff'ın yarısında sadakat yüksek; diğer yarısında hiçbir özellik öne çıkmıyor. Kural tabanlı şapka bu ikinci grubu bilmiyor.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
satirlar = []

for bina, baskin in BINALAR.items():
    for i in range(60):
        if bina == "Hufflepuff" and i % 2 == 1:
            # Dengeli öğrenci: hiçbir özellik öne çıkmıyor
            ogr = {o: rng.normal(5, 0.7) for o in OZELLIKLER}
            ogr["sadakat"] = rng.normal(5.5, 0.7)
        else:
            ogr = {o: rng.normal(4.5, 1.2) for o in OZELLIKLER}
            ogr[baskin] = rng.normal(8, 1)
        ogr = {o: float(np.clip(round(v, 1), 0, 10)) for o, v in ogr.items()}
        ogr["bina"] = bina
        satirlar.append(ogr)

df = pd.DataFrame(satirlar)
print(df.shape)
df.head()

In [ ]:
df.groupby("bina")[OZELLIKLER].mean().round(1)

## 3. Eğitim ve test verisi

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = df[OZELLIKLER]
y = df["bina"]

X_egitim, X_test, y_egitim, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(len(X_egitim), "eğitim,", len(X_test), "test öğrencisi")

kural_tahmin = [kural_sapka(ogr) for ogr in X_test.to_dict("records")]
print(f"Kural tabanlı şapkanın doğruluğu: {accuracy_score(y_test, kural_tahmin):.0%}")

## 4. Karar ağacı: şapkaya örneklerden öğretmek

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_egitim, y_egitim)

tahmin = model.predict(X_test)
print(f"Karar ağacının doğruluğu: {accuracy_score(y_test, tahmin):.0%}")

In [ ]:
print(export_text(model, feature_names=OZELLIKLER))

## 5. Şapka nerede yanılıyor?

In [ ]:
from sklearn.metrics import confusion_matrix

etiketler = list(BINALAR)
pd.DataFrame(confusion_matrix(y_test, tahmin, labels=etiketler),
             index=[f"gerçek: {b}" for b in etiketler],
             columns=[f"tahmin: {b}" for b in etiketler])

## 6. Ağacı büyütmek her zaman iyi mi?

In [ ]:
derinlikler = [1, 2, 3, 4, 5, 6, 8, 10]
egitim_skor, test_skor = [], []

for d in derinlikler:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_egitim, y_egitim)
    egitim_skor.append(accuracy_score(y_egitim, m.predict(X_egitim)))
    test_skor.append(accuracy_score(y_test, m.predict(X_test)))

for d, e, t in zip(derinlikler, egitim_skor, test_skor):
    print(f"derinlik {d:>2}: eğitim {e:.0%}, test {t:.0%}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(derinlikler, egitim_skor, marker="o", linewidth=2, color="#7A4290")
ax.plot(derinlikler, test_skor, marker="o", linewidth=2, color="#D0602A")
ax.text(derinlikler[-1] + 0.2, egitim_skor[-1], "eğitim", color="#333333", va="center")
ax.text(derinlikler[-1] + 0.2, test_skor[-1], "test", color="#333333", va="center")
ax.set_xlabel("Ağacın derinliği")
ax.set_ylabel("Doğruluk")
ax.set_ylim(0.4, 1.02)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlim(0.5, 11.5)
ax.grid(axis="y", color="#E5E5E5")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Derinlik 4'ten sonra eğitim puanı artıyor, test puanı yerinde sayıyor", loc="left")
plt.tight_layout()
plt.show()

## 7. Berabere kalınca: olasılıklar

In [ ]:
def sapka(cesaret, zeka, sadakat, hirs):
    ogr = pd.DataFrame([{"cesaret": cesaret, "zeka": zeka, "sadakat": sadakat, "hirs": hirs}])
    olasiliklar = model.predict_proba(ogr)[0]
    for bina, p in sorted(zip(model.classes_, olasiliklar), key=lambda x: -x[1]):
        if p > 0:
            print(f"{bina:<11} {p:.0%}")
    print("Şapkanın kararı:", model.predict(ogr)[0])

sapka(cesaret=9, zeka=5, sadakat=4, hirs=5)

Cesaret ve hırs eşit olursa ne olur?

In [ ]:
berabere = {"cesaret": 8, "zeka": 5, "sadakat": 5, "hirs": 8}
print("Kural tabanlı şapka:", kural_sapka(berabere))
print()
sapka(**berabere)

## 8. Kendi şapkanı tak

Her soruya 0 ile 10 arasında bir puan ver.

In [ ]:
puanlar = {}
for o in OZELLIKLER:
    puanlar[o] = float(input(f"{o} (0-10): "))

sapka(**puanlar)

## Alıştırmalar

1. `max_depth` değerini 2 ve 5 yapıp doğruluğun nasıl değiştiğine bak.
2. `rng = np.random.default_rng(42)` satırındaki 42'yi değiştir. Sonuçlar ne kadar oynuyor?
3. Veriye beşinci bir özellik ekle (örneğin `merak`) ve hangi binada yüksek olacağına sen karar ver.
4. `DecisionTreeClassifier` yerine `from sklearn.neighbors import KNeighborsClassifier` ile `KNeighborsClassifier(n_neighbors=5)` dene.